In [1]:
import os
import math
import h3
import numpy as np
import pandas as pd
import geopandas as gpd
import torch
from shapely.geometry import Point

PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
RAW_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

# 1. Load Data Tensors & Static Metadata
tensors_npz = np.load(os.path.join(PROCESSED_DATA_DIR, "gnn_feature_tensors.npz"))
X_static = torch.tensor(tensors_npz["X_static"], dtype=torch.float32)
X_dyn = torch.tensor(tensors_npz["X_dyn"], dtype=torch.float32)

df_static = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "h3_static_features.parquet"))
N = len(df_static)
h3_id_to_idx = {h_id: idx for idx, h_id in enumerate(df_static["h3_id"])}

lats = df_static["centroid_lat"].values
lons = df_static["centroid_lon"].values
elevations = df_static["elevation_mean_m"].values

# 2. Build Hexagonal Adjacency Graph and Edge Attributes
def haversine_and_azimuth(lat1, lon1, lat2, lon2):
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    
    a = math.sin(dphi/2.0)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2.0)**2
    c = 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))
    dist_m = 6371000.0 * c
    
    y = math.sin(dlam) * math.cos(phi2)
    x = math.cos(phi1) * math.sin(phi2) - math.sin(phi1) * math.cos(phi2) * math.cos(dlam)
    az_rad = (math.atan2(y, x) + 2.0 * math.pi) % (2.0 * math.pi)
    
    return dist_m, az_rad

src_list = []
dst_list = []
edge_attrs = []

for i in range(N):
    h_str = h3.int_to_str(df_static["h3_id"].iloc[i])
    disk = h3.grid_disk(h_str, 1)
    
    for nbr_str in disk:
        if nbr_str == h_str:
            continue
        nbr_id = np.uint64(h3.str_to_int(nbr_str))
        if nbr_id in h3_id_to_idx:
            j = h3_id_to_idx[nbr_id]
            dist_m, az_rad = haversine_and_azimuth(lats[i], lons[i], lats[j], lons[j])
            slope_delta = (elevations[j] - elevations[i]) / max(1.0, dist_m)
            
            src_list.append(i)
            dst_list.append(j)
            # Edge attributes: [normalized_dist, azimuth_rad / (2*pi), slope_delta]
            edge_attrs.append([dist_m / 1000.0, az_rad / (2.0 * math.pi), slope_delta])

edge_index = torch.tensor([src_list, dst_list], dtype=torch.long)
edge_attr = torch.tensor(edge_attrs, dtype=torch.float32)

print(f"H3 Hexagonal Mesh: {N} nodes, {edge_index.shape[1]} directed edges.")
print(f"Edge index shape: {edge_index.shape} | Edge attr shape: {edge_attr.shape}")

# 3. Ground Truth Observation Mask & Labels (MODIS + WFIGS)
wfigs_path = os.path.join(RAW_DATA_DIR, "wfigs", "wfigs_perimeters.geojson")
gdf_wfigs = gpd.read_file(wfigs_path)

gdf_h3 = gpd.GeoDataFrame(
    df_static[["h3_id"]],
    geometry=[Point(lo, la) for lo, la in zip(lons, lats)],
    crs="EPSG:4326"
)

joined_wfigs = gpd.sjoin(gdf_h3, gdf_wfigs, how="inner", predicate="intersects")
ignition_ids = set(joined_wfigs["h3_id"].unique())

modis_path = os.path.join(RAW_DATA_DIR, "modis", "modis_2023_usa.csv")
df_modis = pd.read_csv(modis_path)
df_modis_aoi = df_modis[
    (df_modis["longitude"] >= -121.8) & (df_modis["longitude"] <= -121.3) &
    (df_modis["latitude"] >= 39.6) & (df_modis["latitude"] <= 40.0) &
    (df_modis["confidence"] >= 50)
]

for _, r in df_modis_aoi.iterrows():
    h_str = h3.latlng_to_cell(r["latitude"], r["longitude"], 8)
    h_u64 = np.uint64(h3.str_to_int(h_str))
    if h_u64 in h3_id_to_idx:
        ignition_ids.add(h_u64)

ignition_indices = [h3_id_to_idx[h_id] for h_id in ignition_ids]

T = 12
y_obs_mask = torch.zeros((T, N), dtype=torch.float32)
y_obs_val = torch.zeros((T, N), dtype=torch.float32)

# Mark confirmed burning cells across all horizons
for idx in ignition_indices:
    y_obs_mask[:, idx] = 1.0
    y_obs_val[:, idx] = 1.0

# Mark non-burnable cells (water/urban/barren) as confirmed unburned
nonburn_indices = np.where(df_static["fuel_model"].values == 99)[0]
for idx in nonburn_indices:
    y_obs_mask[:, idx] = 1.0
    y_obs_val[:, idx] = 0.0

print(f"Supervised Observation Mask: {int(y_obs_mask[0].sum())} supervised nodes ({len(ignition_indices)} ignited, {len(nonburn_indices)} non-burnable).")

# 4. Analytical Rothermel Regularization Limits (from Version A rothermel engine)
from aerospatium.ca_engine.rothermel import rothermel_forward_ros

slopes_rad = np.radians(df_static["slope_mean_deg"].values)
aspects_rad = df_static["aspect_mean_rad"].values
fuel_models = df_static["fuel_model"].values

# Compute physical ROS capacity for each node over 12 hours (m/min -> normalized fraction/hour)
rothermel_ros = np.zeros((T, N), dtype=np.float32)

for t in range(T):
    u10 = X_dyn[t, :, 0].numpy()
    v10 = X_dyn[t, :, 1].numpy()
    ws = np.sqrt(u10**2 + v10**2)
    wd = np.mod(np.arctan2(-u10, -v10) + 2*np.pi, 2*np.pi)
    
    for i in range(N):
        if fuel_models[i] == 99:
            rothermel_ros[t, i] = 0.0
        else:
            ros_val = rothermel_forward_ros(
                int(fuel_models[i]),
                float(slopes_rad[i]),
                float(aspects_rad[i]),
                float(ws[i]),
                float(wd[i]),
                float(aspects_rad[i]), # max spread direction
                fuel_moisture=0.08
            )
            # Normalize: fraction of cell diameter (~932 m) traversed in 1 hour
            # (ros in m/min * 60 min) / 932 m
            cell_frac_per_hour = (ros_val * 60.0) / 932.0
            rothermel_ros[t, i] = min(1.0, cell_frac_per_hour)

rothermel_ros_tensor = torch.tensor(rothermel_ros, dtype=torch.float32)
print(f"Rothermel physical bounds shape: {rothermel_ros_tensor.shape} | Mean max flux: {rothermel_ros_tensor.mean():.4f}")

H3 Hexagonal Mesh: 2526 nodes, 14736 directed edges.
Edge index shape: torch.Size([2, 14736]) | Edge attr shape: torch.Size([14736, 3])
Supervised Observation Mask: 337 supervised nodes (337 ignited, 0 non-burnable).


ModuleNotFoundError: No module named 'aerospatium'

In [2]:
import os
import sys
import math
import h3
import numpy as np
import pandas as pd
import geopandas as gpd
import torch
from shapely.geometry import Point

PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
src_path = os.path.join(PROJECT_ROOT, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

RAW_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

# 1. Load Data Tensors & Static Metadata
tensors_npz = np.load(os.path.join(PROCESSED_DATA_DIR, "gnn_feature_tensors.npz"))
X_static = torch.tensor(tensors_npz["X_static"], dtype=torch.float32)
X_dyn = torch.tensor(tensors_npz["X_dyn"], dtype=torch.float32)

df_static = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "h3_static_features.parquet"))
N = len(df_static)
h3_id_to_idx = {h_id: idx for idx, h_id in enumerate(df_static["h3_id"])}

lats = df_static["centroid_lat"].values
lons = df_static["centroid_lon"].values
elevations = df_static["elevation_mean_m"].values

# 2. Build Hexagonal Adjacency Graph and Edge Attributes
def haversine_and_azimuth(lat1, lon1, lat2, lon2):
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    
    a = math.sin(dphi/2.0)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2.0)**2
    c = 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))
    dist_m = 6371000.0 * c
    
    y = math.sin(dlam) * math.cos(phi2)
    x = math.cos(phi1) * math.sin(phi2) - math.sin(phi1) * math.cos(phi2) * math.cos(dlam)
    az_rad = (math.atan2(y, x) + 2.0 * math.pi) % (2.0 * math.pi)
    
    return dist_m, az_rad

src_list = []
dst_list = []
edge_attrs = []

for i in range(N):
    h_str = h3.int_to_str(df_static["h3_id"].iloc[i])
    disk = h3.grid_disk(h_str, 1)
    
    for nbr_str in disk:
        if nbr_str == h_str:
            continue
        nbr_id = np.uint64(h3.str_to_int(nbr_str))
        if nbr_id in h3_id_to_idx:
            j = h3_id_to_idx[nbr_id]
            dist_m, az_rad = haversine_and_azimuth(lats[i], lons[i], lats[j], lons[j])
            slope_delta = (elevations[j] - elevations[i]) / max(1.0, dist_m)
            
            src_list.append(i)
            dst_list.append(j)
            edge_attrs.append([dist_m / 1000.0, az_rad / (2.0 * math.pi), slope_delta])

edge_index = torch.tensor([src_list, dst_list], dtype=torch.long)
edge_attr = torch.tensor(edge_attrs, dtype=torch.float32)

print(f"H3 Hexagonal Mesh: {N} nodes, {edge_index.shape[1]} directed edges.")
print(f"Edge index shape: {edge_index.shape} | Edge attr shape: {edge_attr.shape}")

# 3. Ground Truth Observation Mask & Labels (MODIS + WFIGS)
wfigs_path = os.path.join(RAW_DATA_DIR, "wfigs", "wfigs_perimeters.geojson")
gdf_wfigs = gpd.read_file(wfigs_path)

gdf_h3 = gpd.GeoDataFrame(
    df_static[["h3_id"]],
    geometry=[Point(lo, la) for lo, la in zip(lons, lats)],
    crs="EPSG:4326"
)

joined_wfigs = gpd.sjoin(gdf_h3, gdf_wfigs, how="inner", predicate="intersects")
ignition_ids = set(joined_wfigs["h3_id"].unique())

modis_path = os.path.join(RAW_DATA_DIR, "modis", "modis_2023_usa.csv")
df_modis = pd.read_csv(modis_path)
df_modis_aoi = df_modis[
    (df_modis["longitude"] >= -121.8) & (df_modis["longitude"] <= -121.3) &
    (df_modis["latitude"] >= 39.6) & (df_modis["latitude"] <= 40.0) &
    (df_modis["confidence"] >= 50)
]

for _, r in df_modis_aoi.iterrows():
    h_str = h3.latlng_to_cell(r["latitude"], r["longitude"], 8)
    h_u64 = np.uint64(h3.str_to_int(h_str))
    if h_u64 in h3_id_to_idx:
        ignition_ids.add(h_u64)

ignition_indices = [h3_id_to_idx[h_id] for h_id in ignition_ids]

T = 12
y_obs_mask = torch.zeros((T, N), dtype=torch.float32)
y_obs_val = torch.zeros((T, N), dtype=torch.float32)

for idx in ignition_indices:
    y_obs_mask[:, idx] = 1.0
    y_obs_val[:, idx] = 1.0

print(f"Supervised Observation Mask: {int(y_obs_mask[0].sum())} supervised nodes ({len(ignition_indices)} ignited).")

# 4. Analytical Rothermel Regularization Limits
from aerospatium.ca_engine.rothermel import rothermel_forward_ros

slopes_rad = np.radians(df_static["slope_mean_deg"].values)
aspects_rad = df_static["aspect_mean_rad"].values
fuel_models = df_static["fuel_model"].values

rothermel_ros = np.zeros((T, N), dtype=np.float32)

for t in range(T):
    u10 = X_dyn[t, :, 0].numpy()
    v10 = X_dyn[t, :, 1].numpy()
    ws = np.sqrt(u10**2 + v10**2)
    wd = np.mod(np.arctan2(-u10, -v10) + 2*np.pi, 2*np.pi)
    
    for i in range(N):
        if fuel_models[i] == 99:
            rothermel_ros[t, i] = 0.0
        else:
            ros_val = rothermel_forward_ros(
                fuel_model=int(fuel_models[i]),
                slope_rad=float(slopes_rad[i]),
                aspect_rad=float(aspects_rad[i]),
                wind_speed_ms=float(ws[i]),
                wind_dir_rad=float(wd[i]),
                neighbor_azimuth_rad=float(aspects_rad[i]),
                fuel_moisture=0.08
            )
            cell_frac_per_hour = (ros_val * 60.0) / 932.0
            rothermel_ros[t, i] = min(1.0, cell_frac_per_hour)

rothermel_ros_tensor = torch.tensor(rothermel_ros, dtype=torch.float32)
print(f"Rothermel physical bounds shape: {rothermel_ros_tensor.shape} | Mean max flux: {rothermel_ros_tensor.mean():.4f}")

H3 Hexagonal Mesh: 2526 nodes, 14736 directed edges.
Edge index shape: torch.Size([2, 14736]) | Edge attr shape: torch.Size([14736, 3])
Supervised Observation Mask: 337 supervised nodes (337 ignited).
Rothermel physical bounds shape: torch.Size([12, 2526]) | Mean max flux: 0.1704


In [3]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. SpatioTemporal GNN Model Definition (imported from aerospatium.ml.gnn_hazard)
from aerospatium.ml.gnn_hazard import (
    HAS_PYG as has_pyg,
    NativeEdgeGAT,
    SpatioTemporalGNNSpread,
    PhysicsInformedSpreadLoss,
)

# 3. Training Loop (100 Epochs on CPU)
torch.manual_seed(42)
model = SpatioTemporalGNNSpread(in_node_dim=15, edge_dim=3, hidden_dim=64)
criterion = PhysicsInformedSpreadLoss(lambda_phys=0.3, lambda_mono=0.8)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

print(f"Engine backend: {'PyTorch Geometric' if has_pyg else 'Native PyTorch Vectorized'}")
print("Training SpatioTemporal GNN (100 Epochs)...")

start_time = time.perf_counter()
for epoch in range(1, 101):
    model.train()
    optimizer.zero_grad()
    
    p_pred = model(X_static, X_dyn, edge_index, edge_attr)
    loss, l_sup, l_phys, l_mono = criterion(p_pred, y_obs_mask, y_obs_val, rothermel_ros_tensor, edge_index)
    
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/100 | Total Loss: {loss.item():.4f} (Sup: {l_sup.item():.4f}, Phys: {l_phys.item():.4f}, Mono: {l_mono.item():.4f})")

total_train_s = time.perf_counter() - start_time
print(f"\nTraining completed in {total_train_s:.2f} seconds ({total_train_s/100*1000:.1f} ms/epoch).")


Engine backend: Native PyTorch Vectorized
Training SpatioTemporal GNN (100 Epochs)...
Epoch   1/100 | Total Loss: 0.6983 (Sup: 0.6978, Phys: 0.0000, Mono: 0.0007)
Epoch  20/100 | Total Loss: 0.0000 (Sup: 0.0000, Phys: 0.0000, Mono: 0.0000)
Epoch  40/100 | Total Loss: 0.0000 (Sup: 0.0000, Phys: 0.0000, Mono: 0.0000)
Epoch  60/100 | Total Loss: 0.0000 (Sup: 0.0000, Phys: 0.0000, Mono: 0.0000)
Epoch  80/100 | Total Loss: 0.0000 (Sup: 0.0000, Phys: 0.0000, Mono: 0.0000)
Epoch 100/100 | Total Loss: 0.0000 (Sup: 0.0000, Phys: 0.0000, Mono: 0.0000)

Training completed in 86.84 seconds (868.4 ms/epoch).


In [4]:
import os
import torch
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# 1. Forward Inference in Evaluation Mode
model.eval()
with torch.no_grad():
    p_pred_all = model(X_static, X_dyn, edge_index, edge_attr).numpy() # [12, 2526]

# Extract discrete forecast horizons
# Index 0 -> T=0h, Index 2 -> T=3h, Index 5 -> T=6h, Index 11 -> T=12h
burn_prob_t0 = p_pred_all[0]
burn_prob_t3 = p_pred_all[2]
burn_prob_t6 = p_pred_all[5]
burn_prob_t12 = p_pred_all[11]

# 2. Save Trained PyTorch Model
models_dir = os.path.join(PROJECT_ROOT, "models")
os.makedirs(models_dir, exist_ok=True)
gnn_model_path = os.path.join(models_dir, "spatiotemporal_gnn.pt")
torch.save(model.state_dict(), gnn_model_path)
print(f"Saved PyTorch GNN model weights to: {gnn_model_path}")

# 3. Assemble and Export Hazard Parquet Table
df_hazard = pd.DataFrame({
    "h3_id": df_static["h3_id"],
    "centroid_lon": df_static["centroid_lon"],
    "centroid_lat": df_static["centroid_lat"],
    "burn_prob_0h": burn_prob_t0.astype(np.float32),
    "burn_prob_3h": burn_prob_t3.astype(np.float32),
    "burn_prob_6h": burn_prob_t6.astype(np.float32),
    "burn_prob_12h": burn_prob_t12.astype(np.float32)
})

hazard_parquet_path = os.path.join(PROCESSED_DATA_DIR, "h3_hazard_t12.parquet")
table = pa.Table.from_pandas(df_hazard)
pq.write_table(table, hazard_parquet_path, compression="snappy")

file_size_kb = os.path.getsize(hazard_parquet_path) / 1024.0

print("\n=======================================================")
print(f"SUCCESS: Exported 12-Hour GNN Hazard Parquet Table")
print(f"Path: {hazard_parquet_path}")
print(f"Rows: {len(df_hazard)}, File Size: {file_size_kb:.2f} KB")
print("=======================================================")
print(f"Mean AOI Burn Prob @ 0h:  {burn_prob_t0.mean():.4f}")
print(f"Mean AOI Burn Prob @ 3h:  {burn_prob_t3.mean():.4f}")
print(f"Mean AOI Burn Prob @ 6h:  {burn_prob_t6.mean():.4f}")
print(f"Mean AOI Burn Prob @ 12h: {burn_prob_t12.mean():.4f}")
print("\nFirst 5 entries:")
print(df_hazard.head(5))

Saved PyTorch GNN model weights to: C:\Users\aseem\OneDrive\Dokumen\aerospatium\models\spatiotemporal_gnn.pt

SUCCESS: Exported 12-Hour GNN Hazard Parquet Table
Path: C:\Users\aseem\OneDrive\Dokumen\aerospatium\data\processed\h3_hazard_t12.parquet
Rows: 2526, File Size: 48.12 KB
Mean AOI Burn Prob @ 0h:  1.0000
Mean AOI Burn Prob @ 3h:  1.0000
Mean AOI Burn Prob @ 6h:  1.0000
Mean AOI Burn Prob @ 12h: 1.0000

First 5 entries:
                h3_id  centroid_lon  centroid_lat  burn_prob_0h  burn_prob_3h  \
0  613194612432961535   -121.300552     39.996578      0.999995      0.999998   
1  613194612441350143   -121.303757     39.988480      0.999995      0.999998   
2  613194612456030207   -121.321999     39.991917      0.999994      0.999998   
3  613194612458127359   -121.329514     39.997684      0.999995      0.999998   
4  613194612460224511   -121.311272     39.994247      0.999995      0.999999   

   burn_prob_6h  burn_prob_12h  
0      0.999999       0.999999  
1      0.999999  

In [5]:
import os
import sys
import time
import math
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
src_path = os.path.join(PROJECT_ROOT, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

# 1. Load Tensors
tensors_npz = np.load(os.path.join(PROCESSED_DATA_DIR, "gnn_feature_tensors.npz"))
X_static = torch.tensor(tensors_npz["X_static"], dtype=torch.float32)
X_dyn = torch.tensor(tensors_npz["X_dyn"], dtype=torch.float32)
df_static = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "h3_static_features.parquet"))
N = len(df_static)
T = 12

# 2. Build Proper Semi-Supervised Ground Truth Masks across Horizons
# Ignition cells: confirmed burned (y=1) from t=0 onwards
# Distant cells (>20km from ignition): confirmed unburned (y=0) at t=0, t=3, t=6
ignition_indices = [idx for idx, h_id in enumerate(df_static["h3_id"]) if h_id in ignition_ids]
ign_lats = df_static["centroid_lat"].iloc[ignition_indices].values
ign_lons = df_static["centroid_lon"].iloc[ignition_indices].values
center_lat, center_lon = np.mean(ign_lats), np.mean(ign_lons)

dists_to_center_km = np.sqrt(
    ((df_static["centroid_lat"].values - center_lat) * 111.0)**2 +
    ((df_static["centroid_lon"].values - center_lon) * 111.0 * np.cos(np.radians(center_lat)))**2
)

unburned_distant_indices = np.where(dists_to_center_km > 18.0)[0]

y_obs_mask = torch.zeros((T, N), dtype=torch.float32)
y_obs_val = torch.zeros((T, N), dtype=torch.float32)

# Ignition supervision
for idx in ignition_indices:
    y_obs_mask[:, idx] = 1.0
    y_obs_val[:, idx] = 1.0

# Negative distant supervision (especially in early horizons)
for idx in unburned_distant_indices:
    y_obs_mask[:6, idx] = 1.0
    y_obs_val[:6, idx] = 0.0

print(f"Balanced Supervised Mask: {len(ignition_indices)} positive nodes, {len(unburned_distant_indices)} negative nodes.")

# 3. SpatioTemporal Diffusion GNN Model
class SpatioTemporalGNNSpread(nn.Module):
    def __init__(self, in_node_dim=15, edge_dim=3, hidden_dim=64):
        super().__init__()
        self.gat1 = NativeEdgeGAT(in_node_dim, hidden_dim, edge_dim=edge_dim)
        self.gat2 = NativeEdgeGAT(hidden_dim, hidden_dim, edge_dim=edge_dim)
        self.recurrent_cell = nn.GRUCell(hidden_dim, hidden_dim)
        
        self.hazard_head = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.SiLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        self.dropout = nn.Dropout(p=0.05)

    def forward(self, x_static, x_dyn_seq, edge_index, edge_attr, init_ignition_mask):
        T, N, _ = x_dyn_seq.shape
        # Initialize internal state conditioned on ignition points
        h_t = torch.zeros(N, 64, device=x_static.device)
        h_t[:, 0] = init_ignition_mask # inject initial burn source
        
        burn_probs = []
        p_prev = init_ignition_mask.clone()

        for t in range(T):
            x_t = torch.cat([x_static, x_dyn_seq[t]], dim=-1)
            
            g_t = self.dropout(torch.relu(self.gat1(x_t, edge_index, edge_attr)))
            g_t = self.dropout(torch.relu(self.gat2(g_t, edge_index, edge_attr)))
            
            h_t = self.recurrent_cell(g_t, h_t)
            
            # Predict marginal arrival hazard
            delta_p = self.hazard_head(h_t).squeeze(-1) * 0.35 # bounded spread per step
            p_t = torch.clamp(p_prev + delta_p, 0.0, 1.0)
            burn_probs.append(p_t)
            p_prev = p_t

        return torch.stack(burn_probs, dim=0)

# Initial ignition state tensor
init_ign = torch.zeros(N, dtype=torch.float32)
init_ign[ignition_indices] = 1.0

# 4. Refined Physics-Informed Loss with Sparsity/Energy Regularization
class PhysicsInformedSpreadLoss(nn.Module):
    def __init__(self, lambda_phys=0.2, lambda_mono=0.5, lambda_sparse=0.05):
        super().__init__()
        self.bce = nn.BCELoss(reduction='none')
        self.lambda_phys = lambda_phys
        self.lambda_mono = lambda_mono
        self.lambda_sparse = lambda_sparse

    def forward(self, p_pred, y_obs_mask, y_obs_val, rothermel_ros, edge_index):
        unreduced_bce = self.bce(p_pred, y_obs_val)
        l_sup = (unreduced_bce * y_obs_mask).sum() / (y_obs_mask.sum() + 1e-6)

        # Spatial diffusion flux bounded by Rothermel
        src, dst = edge_index
        spatial_flux = torch.abs(p_pred[:-1, src] - p_pred[:-1, dst])
        l_phys = torch.mean(torch.relu(spatial_flux - rothermel_ros[:-1, src]))

        # Monotonicity
        l_mono = torch.mean(torch.relu(p_pred[:-1, :] - p_pred[1:, :]))
        
        # Sparsity penalty (prevents global fire explosion)
        l_sparse = torch.mean(p_pred)

        total_loss = l_sup + (self.lambda_phys * l_phys) + (self.lambda_mono * l_mono) + (self.lambda_sparse * l_sparse)
        return total_loss, l_sup, l_phys, l_mono, l_sparse

# 5. Retrain Model
torch.manual_seed(42)
model = SpatioTemporalGNNSpread(in_node_dim=15, edge_dim=3, hidden_dim=64)
criterion = PhysicsInformedSpreadLoss(lambda_phys=0.2, lambda_mono=0.5, lambda_sparse=0.05)
optimizer = torch.optim.Adam(model.parameters(), lr=0.008)

print("\nRetraining SpatioTemporal GNN with Boundary Regularization (60 Epochs)...")
start_time = time.perf_counter()

for epoch in range(1, 61):
    model.train()
    optimizer.zero_grad()
    
    p_pred = model(X_static, X_dyn, edge_index, edge_attr, init_ign)
    loss, l_sup, l_phys, l_mono, l_sp = criterion(p_pred, y_obs_mask, y_obs_val, rothermel_ros_tensor, edge_index)
    
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:2d}/60 | Loss: {loss.item():.4f} (Sup: {l_sup.item():.4f}, Phys: {l_phys.item():.4f}, Mono: {l_mono.item():.4f}, Sparse: {l_sp.item():.4f})")

total_train_s = time.perf_counter() - start_time
print(f"Training completed in {total_train_s:.2f} seconds.")

# 6. Re-export Validated Hazard Predictions
model.eval()
with torch.no_grad():
    p_pred_all = model(X_static, X_dyn, edge_index, edge_attr, init_ign).numpy()

burn_prob_t0 = p_pred_all[0]
burn_prob_t3 = p_pred_all[2]
burn_prob_t6 = p_pred_all[5]
burn_prob_t12 = p_pred_all[11]

df_hazard = pd.DataFrame({
    "h3_id": df_static["h3_id"],
    "centroid_lon": df_static["centroid_lon"],
    "centroid_lat": df_static["centroid_lat"],
    "burn_prob_0h": burn_prob_t0.astype(np.float32),
    "burn_prob_3h": burn_prob_t3.astype(np.float32),
    "burn_prob_6h": burn_prob_t6.astype(np.float32),
    "burn_prob_12h": burn_prob_t12.astype(np.float32)
})

hazard_parquet_path = os.path.join(PROCESSED_DATA_DIR, "h3_hazard_t12.parquet")
table = pa.Table.from_pandas(df_hazard)
pq.write_table(table, hazard_parquet_path, compression="snappy")

print("\n--- Calibrated GNN Spread Results across AOI ---")
print(f"Mean Burn Prob @ 0h:  {burn_prob_t0.mean()*100:.2f} %")
print(f"Mean Burn Prob @ 3h:  {burn_prob_t3.mean()*100:.2f} %")
print(f"Mean Burn Prob @ 6h:  {burn_prob_t6.mean()*100:.2f} %")
print(f"Mean Burn Prob @ 12h: {burn_prob_t12.mean()*100:.2f} %")
print("\nFirst 5 entries:")
print(df_hazard.head(5))

Balanced Supervised Mask: 337 positive nodes, 1180 negative nodes.

Retraining SpatioTemporal GNN with Boundary Regularization (60 Epochs)...
Epoch  1/60 | Loss: 11.7976 (Sup: 11.7553, Phys: 0.0046, Mono: 0.0000, Sparse: 0.8272)
Epoch 10/60 | Loss: 0.7182 (Sup: 0.7064, Phys: 0.0258, Mono: 0.0000, Sparse: 0.1339)
Epoch 20/60 | Loss: 0.7180 (Sup: 0.7061, Phys: 0.0258, Mono: 0.0000, Sparse: 0.1334)
Epoch 30/60 | Loss: 0.7180 (Sup: 0.7061, Phys: 0.0258, Mono: 0.0000, Sparse: 0.1334)
Epoch 40/60 | Loss: 0.7180 (Sup: 0.7061, Phys: 0.0258, Mono: 0.0000, Sparse: 0.1334)
Epoch 50/60 | Loss: 0.7180 (Sup: 0.7061, Phys: 0.0258, Mono: 0.0000, Sparse: 0.1335)
Epoch 60/60 | Loss: 0.7180 (Sup: 0.7061, Phys: 0.0258, Mono: 0.0000, Sparse: 0.1334)
Training completed in 50.25 seconds.

--- Calibrated GNN Spread Results across AOI ---
Mean Burn Prob @ 0h:  13.34 %
Mean Burn Prob @ 3h:  13.34 %
Mean Burn Prob @ 6h:  13.34 %
Mean Burn Prob @ 12h: 13.34 %

First 5 entries:
                h3_id  centroid_lon 

In [6]:
import os
import sys
import time
import math
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
models_dir = os.path.join(PROJECT_ROOT, "models")
os.makedirs(models_dir, exist_ok=True)

# 1. Load Data
tensors_npz = np.load(os.path.join(PROCESSED_DATA_DIR, "gnn_feature_tensors.npz"))
X_static = torch.tensor(tensors_npz["X_static"], dtype=torch.float32)
X_dyn = torch.tensor(tensors_npz["X_dyn"], dtype=torch.float32)

df_static = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "h3_static_features.parquet"))
df_ca = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "ca_simulation_camp_fire.parquet"))
N = len(df_static)
T = 12

arr_times = df_ca["predicted_arrival_min"].values

# Continuous ground-truth probability targets per horizon
y_targets = torch.zeros((T, N), dtype=torch.float32)
for t in range(T):
    horizon_min = (t + 1) * 60.0
    # Sigmoid smooth transition around arrival time
    diff = (horizon_min - arr_times) / 30.0
    y_targets[t] = torch.tensor(1.0 / (1.0 + np.exp(-np.clip(diff, -10.0, 10.0))), dtype=torch.float32)

print(f"Target burn progression:")
print(f"T=1h:  {y_targets[0].mean()*100:.1f} % | T=3h:  {y_targets[2].mean()*100:.1f} %")
print(f"T=6h:  {y_targets[5].mean()*100:.1f} % | T=12h: {y_targets[11].mean()*100:.1f} %")

# 2. Resilient SpatioTemporal GNN with State Skip-Connections
class ResilientEdgeGAT(nn.Module):
    def __init__(self, in_dim, out_dim, edge_dim=3):
        super().__init__()
        self.lin_src = nn.Linear(in_dim, out_dim, bias=False)
        self.lin_dst = nn.Linear(in_dim, out_dim, bias=False)
        self.lin_edge = nn.Linear(edge_dim, out_dim, bias=False)
        self.att = nn.Parameter(torch.Tensor(1, out_dim))
        self.leaky_relu = nn.LeakyReLU(0.1)
        nn.init.xavier_uniform_(self.att)

    def forward(self, x, edge_index, edge_attr):
        src, dst = edge_index
        h_src = self.lin_src(x[src])
        h_dst = self.lin_dst(x[dst])
        h_e = self.lin_edge(edge_attr)
        
        alpha = self.leaky_relu((h_src + h_dst + h_e) * self.att).sum(dim=-1, keepdim=True)
        alpha = torch.exp(alpha - alpha.max())
        alpha_sum = torch.zeros(x.size(0), 1, device=x.device).index_add_(0, dst, alpha) + 1e-6
        alpha_norm = alpha / alpha_sum[dst]
        
        msg = (h_src + h_e) * alpha_norm
        out = torch.zeros(x.size(0), h_src.size(-1), device=x.device).index_add_(0, dst, msg)
        return out

class SpatioTemporalGNNSpread(nn.Module):
    def __init__(self, in_node_dim=15, edge_dim=3, hidden_dim=64):
        super().__init__()
        self.gat1 = ResilientEdgeGAT(in_node_dim, hidden_dim, edge_dim=edge_dim)
        self.gat2 = ResilientEdgeGAT(hidden_dim, hidden_dim, edge_dim=edge_dim)
        self.state_proj = nn.Linear(1, hidden_dim)
        self.recurrent_cell = nn.GRUCell(hidden_dim, hidden_dim)
        self.hazard_head = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.LeakyReLU(0.1),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x_static, x_dyn_seq, edge_index, edge_attr, p_init):
        T, N, _ = x_dyn_seq.shape
        h_t = self.state_proj(p_init.unsqueeze(-1))
        p_prev = p_init
        burn_probs = []

        for t in range(T):
            x_t = torch.cat([x_static, x_dyn_seq[t]], dim=-1)
            g_t = torch.relu(self.gat1(x_t, edge_index, edge_attr))
            g_t = torch.relu(self.gat2(g_t, edge_index, edge_attr))
            
            # Recurrent update with previous burn state injection
            h_t = self.recurrent_cell(g_t + self.state_proj(p_prev.unsqueeze(-1)), h_t)
            p_t = self.hazard_head(h_t).squeeze(-1)
            
            # Enforce temporal monotonic expansion
            p_t = torch.maximum(p_t, p_prev)
            burn_probs.append(p_t)
            p_prev = p_t

        return torch.stack(burn_probs, dim=0)

# Initial ignition state tensor at T=0
p_init = torch.tensor((arr_times == 0.0).astype(np.float32))

# 3. Model Training
torch.manual_seed(42)
model = SpatioTemporalGNNSpread(in_node_dim=15, edge_dim=3, hidden_dim=64)
optimizer = torch.optim.Adam(model.parameters(), lr=0.008, weight_decay=1e-5)
mse_loss = nn.MSELoss()

print("\n--- Training Calibrated GNN with State Injection (80 Epochs) ---")
start_time = time.perf_counter()

for epoch in range(1, 81):
    model.train()
    optimizer.zero_grad()
    
    p_pred = model(X_static, X_dyn, edge_index, edge_attr, p_init)
    loss = mse_loss(p_pred, y_targets)
    
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:2d}/80 | MSE Loss: {loss.item():.6f}")

print(f"Training completed in {time.perf_counter() - start_time:.2f} seconds.")

# 4. Generate Predictions & Export
model.eval()
with torch.no_grad():
    p_pred_final = model(X_static, X_dyn, edge_index, edge_attr, p_init).numpy()

burn_prob_t0 = p_init.numpy()
burn_prob_t3 = p_pred_final[2]
burn_prob_t6 = p_pred_final[5]
burn_prob_t12 = p_pred_final[11]

gnn_model_path = os.path.join(models_dir, "spatiotemporal_gnn.pt")
torch.save(model.state_dict(), gnn_model_path)

df_hazard = pd.DataFrame({
    "h3_id": df_static["h3_id"],
    "centroid_lon": df_static["centroid_lon"],
    "centroid_lat": df_static["centroid_lat"],
    "burn_prob_0h": burn_prob_t0.astype(np.float32),
    "burn_prob_3h": burn_prob_t3.astype(np.float32),
    "burn_prob_6h": burn_prob_t6.astype(np.float32),
    "burn_prob_12h": burn_prob_t12.astype(np.float32)
})

hazard_parquet_path = os.path.join(PROCESSED_DATA_DIR, "h3_hazard_t12.parquet")
df_hazard.to_parquet(hazard_parquet_path, compression="snappy")

print("\n=======================================================")
print(f"SUCCESS: Calibrated 12-Hour GNN Hazard Output")
print(f"Path: {hazard_parquet_path}")
print("=======================================================")
print(f"Mean AOI Burn Prob @ 0h:  {burn_prob_t0.mean()*100:.1f} %")
print(f"Mean AOI Burn Prob @ 3h:  {burn_prob_t3.mean()*100:.1f} %")
print(f"Mean AOI Burn Prob @ 6h:  {burn_prob_t6.mean()*100:.1f} %")
print(f"Mean AOI Burn Prob @ 12h: {burn_prob_t12.mean()*100:.1f} %")
print("\nSample Preview:")
print(df_hazard.head(5))

Target burn progression:
T=1h:  17.9 % | T=3h:  33.9 %
T=6h:  53.0 % | T=12h: 78.0 %

--- Training Calibrated GNN with State Injection (80 Epochs) ---
Epoch  1/80 | MSE Loss: 0.196934
Epoch 20/80 | MSE Loss: 0.121528
Epoch 40/80 | MSE Loss: 0.096810
Epoch 60/80 | MSE Loss: 0.079442
Epoch 80/80 | MSE Loss: 0.072976
Training completed in 67.59 seconds.

SUCCESS: Calibrated 12-Hour GNN Hazard Output
Path: C:\Users\aseem\OneDrive\Dokumen\aerospatium\data\processed\h3_hazard_t12.parquet
Mean AOI Burn Prob @ 0h:  13.3 %
Mean AOI Burn Prob @ 3h:  33.8 %
Mean AOI Burn Prob @ 6h:  58.3 %
Mean AOI Burn Prob @ 12h: 78.5 %

Sample Preview:
                h3_id  centroid_lon  centroid_lat  burn_prob_0h  burn_prob_3h  \
0  613194612432961535   -121.300552     39.996578           0.0      0.051734   
1  613194612441350143   -121.303757     39.988480           0.0      0.047360   
2  613194612456030207   -121.321999     39.991917           0.0      0.056558   
3  613194612458127359   -121.329514     